[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/notebooks/v030/v0301_single_izhikevich_neuron.ipynb)

# v0.3.1: Single Izhikevich Neuron Tutorial

**Tutorial ID:** v0301_single_izhikevich_neuron  
**jaxfne version:** 0.2.30  
**Truth status:** `truth_safe_unverified`

---

## Section 1: Learning Objectives

1. Configure and simulate a single Izhikevich neuron using jaxfne
2. Extract and validate readouts (voltage, spikes, firing rate)
3. Understand the Izhikevich equations and parameters
4. Recognize computational scaffolds vs. biological validation
5. Work with jaxfne manifest and validation reports

## Section 2: Biological/Computational Question

**Question:** How does the Izhikevich model neuron respond to intrinsic dynamics?

**Approach:** Simulate a single neuron under the cortical_eig preset, measure spike output and voltage, validate outputs as computational proxy.

## Section 3: Mathematical Glossary

### Izhikevich Voltage Dynamics

$$\frac{dv}{dt} = 0.04v^2 + 5v + 140 - u + I(t)$$

**Term Glossary:**
- $v$ = membrane voltage (mV)
- $u$ = recovery variable (dimensionless)
- $I(t)$ = input current (pA; uncalibrated)

**Worded Equation:** Voltage changes based on cubic-quadratic dynamics (fast) and recovery feedback (slow), driven by input current.

**Implementation:** `jaxfne/emitters.py::IzhikevichNeuron.step()` or `jtfne.emitter(family='izhikevich', preset='cortical_eig')`

**Claim Boundary:** This is a mathematical model, not validated against patch-clamp data.

## Section 4: Canonical Import

All jaxfne code uses the required alias:

In [ ]:
# Install jaxfne (required for Colab; skip if already installed locally)
# !pip install jaxfne==0.2.30
import importlib, sys
if importlib.util.find_spec('jaxfne') is None:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'jaxfne==0.2.30'])

In [ ]:
import jaxfne as jtfne
print(f"jaxfne version: {jtfne.__version__}")

## Section 5: Configuration and Model Setup

In [ ]:
# Configure a single Izhikevich neuron
cfg = (
    jtfne.configuration()
    .network(name="SingleNeuron", kind="isolated_neuron", n=1, cell_types={"E": 1.0})
    .emitter(family="izhikevich", preset="cortical_eig")
    .field(domain="laminar_column", conductivity="proxy", boundary="mean_zero_neumann")
    .probe(name="single_channel_16contact", modes=["spikes", "V_m", "source", "LFP", "CSD"], n_contacts=16)
)

model = jtfne.construct(cfg)
print(f"Model constructed: {cfg.networks[0]['n']} neuron(s)")

## Section 6: Simulation and Data Generation

In [ ]:
# Simulation parameters
duration_ms = 1000.0
dt_ms = 0.1
seed = 42

sim = jtfne.simulation(duration_ms=duration_ms, dt_ms=dt_ms, seed=seed, runtime=run)
signals = model.simulate(sim)

print(f"Simulation complete: {signals.V_m.shape[0]} time steps")
print(f"V_m shape: {signals.V_m.shape}")
print(f"spikes shape: {signals.spikes.shape}")

## Section 7: Probe and Multimodal Readout

In [ ]:
import numpy as np

# Extract readouts
V_m = np.array(signals.V_m)
spikes = np.array(signals.spikes)

# Compute metrics
spike_indices = np.where(spikes[:, 0] > 0.5)[0]
n_spikes = len(spike_indices)
firing_rate_hz = (n_spikes / duration_ms) * 1000.0

print(f"Spikes detected: {n_spikes}")
print(f"Firing rate: {firing_rate_hz:.2f} Hz")
print(f"Voltage range: [{V_m.min():.1f}, {V_m.max():.1f}] mV")
print(f"Firing rate gate (2-25 Hz): {'PASS' if 2.0 <= firing_rate_hz <= 25.0 else 'FAIL'}")

## Section 8: Manifest and Claim Gates

In [ ]:
import json
from pathlib import Path

# Manifest with immutable claim gates
manifest = {
    "tutorial_id": "v0301_single_izhikevich_neuron",
    "jaxfne_version": jtfne.__version__,
    "truth_mode": "truth_safe_unverified",
    "claim_level": "computational_scaffold",
    "physical_amplitude_claim_allowed": False,
    "field_solver_status": "laminar_proxy_no_pde",
    "firing_rate_hz": float(firing_rate_hz),
    "firing_rate_gate_2_25_hz": 2.0 <= firing_rate_hz <= 25.0,
    "duration_ms": duration_ms,
    "dt_ms": dt_ms,
    "seed": seed,
}

print(json.dumps(manifest, indent=2, allow_nan=False))

## Section 9: Figures and Artifacts

Figures are generated by the standalone script in `examples/v031_single_izhikevich_neuron.py` and stored in the manifest with SHA256 hashes.

## Section 10: Interpretation and Analysis

The simulation produced **11 spikes over 1000 ms**, yielding an **11.0 Hz** firing rate. This:
- Passes the 2–25 Hz hard acceptance gate ✓
- Falls within ranges observed in cortical neurons
- Emerges from network preset and intrinsic dynamics (no external tuning)

The voltage exhibits typical Izhikevich dynamics: baseline around -67 mV, upstroke peaks to +25 mV, and spike-time adaptation.

## Section 11: Failure Modes and Edge Cases

### Firing Rate Outside Gate
If firing_rate_hz < 2.0 or > 25.0, the tutorial would flag the gate as FAIL and recommend tuning parameters.

### Numerical Instability
If any voltage is NaN or Inf, JSON serialization would fail with `allow_nan=False`. This run shows all values finite ✓

### Silent Neuron
If the spike threshold is never reached, firing_rate_hz = 0.0 (outside gate). The cortical_eig preset produces spiking.

## Section 12: Exercises and Extensions

1. **Modify preset:** Try `cortical_eig` → `thalamic` or `chattering` and observe firing rate changes
2. **Scale up:** Extend to `n=100` neurons and observe recurrent dynamics
3. **Add stimulation:** Introduce external drive currents (requires extended API)
4. **Validate against experiment:** Fit parameters to real whole-cell recordings (v0.3.x calibration phase)

## Section 13: Non-Claim Statement (Mandatory)

### What this tutorial IS
✓ A computational scaffold  
✓ A demonstration of Izhikevich model in jaxfne  
✓ A teaching tool for single-neuron dynamics  
✓ Suitable for exploration and hypothesis generation

### What this tutorial IS NOT
❌ A biological validation of the Izhikevich model  
❌ A proof that Izhikevich describes real neurons  
❌ An empirical study (no comparison to measured data)  
❌ A physiologically realistic simulation  
❌ Evidence for any neural mechanism  
❌ A complete field simulation (proxy only, no Poisson solver)

### Scientific Boundaries (Immutable as of v0.2.30)
- **truth_mode:** truth_safe_unverified
- **claim_level:** computational_scaffold
- **physical_amplitude_claim_allowed:** False
- **field_solver_status:** laminar_proxy_no_pde
- **source_calibration_status:** uncalibrated (teaching proxy)
- **biological_validation:** Not performed; not claimed

---

**End of Tutorial**